In [ ]:
import pandas as pd
import json
import os
import webbrowser
from pathlib import Path
BASE_DIR = Path(os.getcwd())
EXCEL_PATH = BASE_DIR / "Events_master.xlsx"
EXCEL_PREP_PATH = BASE_DIR / "Events_preparation.xlsx"
SHP_JS_PATH = BASE_DIR / "ksa_map_simple.js"
OUT_HTML = BASE_DIR / "dashboard_output.html"
ASSETS_DIR = BASE_DIR / "assets"
print(f"📊 مسار ملف التنفيذ: {EXCEL_PATH.name}")
print(f"📝 مسار ملف الإعداد: {EXCEL_PREP_PATH.name}")
print("🔄 حوكمة وتوزيع البيانات + دمج المخرجات المتكررة بحسب الرقم التسلسلي...")

In [ ]:
fix_dict = {
    'منطقة الرياض': 'Ar Riyad', 'الرياض': 'Ar Riyad',
    'المنطقة الشرقية': 'Ash Sharqiyah', 'محافظة الأحساء': 'Ash Sharqiyah', 'الشرقية': 'Ash Sharqiyah',
    'منطقة مكة المكرمة': 'Makkah', 'محافظة جدة': 'Makkah', 'محافظة الطائف': 'Makkah', 'مكة': 'Makkah', 'مكة المكرمة': 'Makkah',
    'منطقة حائل': "Ha'il", 'حائل': "Ha'il",
    'منطقة القصيم': 'Al Quassim', 'القصيم': 'Al Quassim',
    'منطقة المدينة المنورة': 'Al Madinah', 'المدينة المنورة': 'Al Madinah', 'المدينة': 'Al Madinah',
    'منطقة الجوف': 'Al Jawf', 'الجوف': 'Al Jawf',
    'منطقة نجران': 'Najran', 'نجران': 'Najran',
    'منطقة تبوك': 'Tabuk', 'تبوك': 'Tabuk',
    'منطقة الباحة': 'Al Bahah', 'الباحة': 'Al Bahah',
    'منطقة الحدود الشمالية': 'Al Hudud ash Shamaliyah', 'الحدود الشمالية': 'Al Hudud ash Shamaliyah',
    'منطقة عسير': '`Asir', 'عسير': '`Asir',
    'منطقة جازان': 'Jizan', 'جازان': 'Jizan', 'جيزان': 'Jizan'
}

In [ ]:
REGION_TO_ARABIC = {
    'Ar Riyad': 'الرياض', 'Ash Sharqiyah': 'المنطقة الشرقية', 'Makkah': 'مكة المكرمة',
    "Ha'il": 'حائل', 'Al Quassim': 'القصيم', 'Al Madinah': 'المدينة المنورة',
    'Al Jawf': 'الجوف', 'Najran': 'نجران', 'Tabuk': 'تبوك', 'Al Bahah': 'الباحة',
    'Al Hudud ash Shamaliyah': 'الحدود الشمالية', '`Asir': 'عسير', 'Jizan': 'جازان',
    'International': 'دولي'
}

In [ ]:
MAIN_CITIES_MAP = {
    "Ar Riyad": "الرياض", "Makkah": "مكة المكرمة", "Al Madinah": "المدينة المنورة",
    "Ash Sharqiyah": "الدمام", "Tabuk": "تبوك", "Al Quassim": "بريدة",
    "Ha'il": "حائل", "Jizan": "جازان", "`Asir": "أبها", "Najran": "نجران",
    "Al Jawf": "سكاكا", "Al Hudud ash Shamaliyah": "عرعر", "Al Bahah": "الباحة",
    "International": "دولي"
}

In [ ]:
ALL_REGIONS = list(MAIN_CITIES_MAP.keys())

In [ ]:
AVAILABLE_CITIES_BY_REGION = {
    "Ar Riyad": ["الرياض", "الدرعية", "الخرج", "الدوادمي", "المجمعة", "القويعية", "الأفلاج", "وادي الدواسر", "الزلفي", "شقراء", "حوطة بني تميم", "عفيف", "الغاط", "السليل", "ضرما", "المزاحمية", "رماح", "ثادق", "حريملاء", "الحريق", "مرات"],
    "Makkah": ["مكة المكرمة", "جدة", "الطائف", "القنفذة", "الليث", "رابغ", "الجموم", "خليص", "الكامل", "الخرمة", "رنية", "تربة", "العرضيات", "أضم", "ميسان", "المويه"],
    "Al Madinah": ["المدينة المنورة", "ينبع", "العلا", "المهد", "بدر", "خيبر", "الحناكية", "العيص", "وادي الفرع"],
    "Al Quassim": ["بريدة", "عنيزة", "الرس", "المذنب", "البكيرية", "البدائع", "الأسياح", "النبهانية", "عيون الجواء", "رياض الخبراء", "الشماسية", "عقلة الصقور", "ضرية"],
    "Ash Sharqiyah": ["الدمام", "الخبر", "الظهران", "الأحساء", "حفر الباطن", "الجبيل", "القطيف", "الخفجي", "رأس تنورة", "بقيق", "النعيرية", "قرية العليا", "العديد"],
    "`Asir": ["أبها", "خميس مشيط", "بيشة", "النماص", "محايل", "سراة عبيدة", "تثليث", "رجال ألمع", "بلقرن", "أحد رفيدة", "المجاردة", "ظهران الجنوب", "بارق", "تنومة", "البرك", "طريب"],
    "Tabuk": ["تبوك", "الوجه", "ضباء", "تيماء", "أملج", "حقل", "البدع"],
    "Ha'il": ["حائل", "بقعاء", "الغزالة", "الشنان", "سميراء", "موقق", "الحائط", "السليمي", "الشملي"],
    "Al Hudud ash Shamaliyah": ["عرعر", "رفحاء", "طريف", "العويقيلة"],
    "Jizan": ["جازان", "صبيا", "أبو عريش", "صامطة", "الحرث", "ضمد", "الريث", "بيش", "الدائر", "أحد المسارحة", "العارضة", "القياس", "العيدابي", "الدرب", "فيفاء", "الطوال", "هروب"],
    "Najran": ["نجران", "شرورة", "حبونا", "بدر الجنوب", "يدمة", "ثار", "خباش"],
    "Al Bahah": ["الباحة", "بلجرشي", "المندق", "المخواة", "العقيق", "قلوة", "القرى", "الحجرة", "بني حسن", "غامد الزناد"],
    "Al Jawf": ["سكاكا", "القريات", "دومة الجندل", "طبرجل"]
}

In [ ]:
path_colors_map = {}
element_colors_map = {}

Fixed distinct colors for Paths (avoiding standard blues) - Warm Palette
AVAILABLE_PATH_COLORS = [
    "#8B0000", "#A52A2A", "#B22222", "#DC143C", "#FF0000", 
    "#FF4500", "#FF6347", "#FF8C00", "#FFA500", "#D2691E",
    "#CD853F", "#A0522D", "#800000", "#8B4513", "#556B2F"
]
AVAILABLE_ELEMENT_COLORS = [
    "#8B008B", "#9932CC", "#8A2BE2", "#4B0082", "#483D8B",
    "#2F4F4F", "#556B2F", "#6B8E23", "#808000", "#BDB76B",
    "#DAA520", "#B8860B", "#CD5C5C", "#BC8F8F", "#F4A460",
    "#DEB887", "#D2B48C", "#F0E68C", "#EEE8AA", "#BDB76B"
]
AVAILABLE_ELEMENT_COLORS = [
    "#F44336", "#E91E63", "#9C27B0", "#673AB7", "#3F51B5",
    "#00BCD4", "#009688", "#4CAF50", "#8BC34A", "#CDDC39",
    "#FFEB3B", "#FFC107", "#FF9800", "#FF5722", "#795548",
    "#607D8B", "#FF8A65", "#4DD0E1", "#BA68C8", "#81C784"
]

In [ ]:
# Fixed distinct colors for Paths (avoiding standard blues) - Warm Palette
AVAILABLE_PATH_COLORS = [
    "#2E4053", "#8B4513", "#556B2F", "#483D8B", "#800000", 
    "#006400", "#4B0082", "#2F4F4F", "#8B0000", "#191970",
    "#008080", "#8A2BE2", "#A0522D", "#696969", "#5F9EA0"
]
AVAILABLE_ELEMENT_COLORS = [
    "#1E3D59", "#B85C38", "#4A5568", "#2C3E50", "#E67E22",
    "#16A085", "#8E44AD", "#D35400", "#27AE60", "#C0392B",
    "#34495E", "#F39C12", "#7F8C8D", "#9B59B6", "#1ABC9C",
    "#E74C3C", "#2ECC71", "#BDC3C7", "#95A5A6", "#3498DB"
]

path_color_index = 0
element_color_index = 0

In [ ]:
def get_path_color(path_name):
    global path_color_index
    if path_name not in path_colors_map:
        path_colors_map[path_name] = AVAILABLE_PATH_COLORS[path_color_index % len(AVAILABLE_PATH_COLORS)]
        path_color_index += 1
    return path_colors_map[path_name]

In [ ]:
def get_element_color(element_name):
    global element_color_index
    if element_name not in element_colors_map:
        element_colors_map[element_name] = AVAILABLE_ELEMENT_COLORS[element_color_index % len(AVAILABLE_ELEMENT_COLORS)]
        element_color_index += 1
    return element_colors_map[element_name]

In [ ]:
def process_stage_data(file_path):

    try:
        if not Path(file_path).exists(): return None
        df_main = pd.read_excel(str(file_path), sheet_name=0)
        
        id_col = 'الرقم التسلسلي' if 'الرقم التسلسلي' in df_main.columns else df_main.columns[0]
        org_col = 'الجهة المسؤولة' if 'الجهة المسؤولة' in df_main.columns else ('الجهة' if 'الجهة' in df_main.columns else 'الجهة التابعة')
        region_col = 'المنطقة الادارية'
        director_col = 'اسم المخرج' if 'اسم المخرج' in df_main.columns else None
        
        df_clean = df_main.copy()
        df_clean[id_col] = df_clean[id_col].astype(str).str.strip()
        df_clean[org_col] = df_clean[org_col].astype(str).str.strip()
        df_clean[region_col] = df_clean[region_col].astype(str).str.strip()
        if 'العنصر' in df_clean.columns: df_clean['العنصر'] = df_clean['العنصر'].astype(str).str.strip()
        if director_col: df_clean[director_col] = df_clean[director_col].astype(str).str.strip()
            
        df_clean['cleaned_region'] = df_clean[region_col].replace(fix_dict)


        
        expanded_events = []
        for _, row in df_clean.iterrows():
            orig_region = str(row[region_col]).strip()
            city = str(row.get('المدينة', '-')).strip()
            
            # Handling International (أخرى)
            if orig_region == 'أخرى' or orig_region == 'اخرى':
                new_row = row.copy()
                new_row['cleaned_region'] = 'International'
                new_row['المنطقة الادارية'] = 'دولي'
                new_row['المدينة'] = city
                if 'العنصر' in new_row and pd.notna(new_row['العنصر']):
                    new_row['العنصر'] = str(new_row['العنصر']).strip() + " - دولي"
                expanded_events.append(new_row)
                continue
                
            if 'جميع المناطق' in orig_region:
                regions_to_process = ALL_REGIONS
            else:
                regions_to_process = [row['cleaned_region']]
                
            for r in regions_to_process:
                if r not in ALL_REGIONS:
                    continue
                    
                cities_to_add = [city]
                if 'جميع المناطق' in city:
                    cities_to_add = AVAILABLE_CITIES_BY_REGION.get(r, [MAIN_CITIES_MAP.get(r, r)])
                elif 'جميع المدن الرئيس' in city:
                    cities_to_add = [MAIN_CITIES_MAP.get(r, r)]
                
                for c in cities_to_add:
                    new_row = row.copy()
                    new_row['cleaned_region'] = r
                    new_row['المنطقة الادارية'] = r
                    new_row['المدينة'] = c
                    expanded_events.append(new_row)

        
        df_expanded = pd.DataFrame(expanded_events) if expanded_events else df_clean.copy()
        
        subset_national = [id_col]
        if director_col: subset_national.append(director_col)
        if 'العنصر' in df_clean.columns: subset_national.append('العنصر')
        
        subset_regional = subset_national + ['cleaned_region']

        df_unique_national = df_clean.drop_duplicates(subset=subset_national) if len(df_clean) > 0 else df_clean
        df_unique_regional = df_expanded.drop_duplicates(subset=subset_regional) if len(df_expanded) > 0 else df_expanded
        df_unique_expanded = df_expanded.drop_duplicates(subset=[id_col], keep='first') if id_col else df_expanded
        
        counts_dict = {r: 0 for r in ALL_REGIONS}
        counts_dict['International'] = 0
        for r in df_unique_regional['cleaned_region']:
            if r in counts_dict: counts_dict[r] += 1
            
        raw_counts = df_unique_expanded['العنصر'].value_counts().to_dict() if 'العنصر' in df_unique_expanded.columns else {}
        df_unique_national = df_unique_expanded
        
        region_events = {}; sub_charts = {}; element_orgs = {}
        
        for r_en, cities in AVAILABLE_CITIES_BY_REGION.items():
            sub_charts[r_en] = {c: 0 for c in cities}
            
        for _, row in df_expanded.iterrows():
            r = row['cleaned_region']
            el = str(row['العنصر']).strip() if pd.notna(row['العنصر']) else "-"
            org = str(row[org_col]).strip() if pd.notna(row[org_col]) else "جهة غير محددة"
            city = str(row.get('المدينة', '-')).strip() if pd.notna(row.get('المدينة')) else "-"
            director = str(row.get('اسم المخرج', '-')).strip() if pd.notna(row.get('اسم المخرج')) else "-"
            desc = str(row.get('الوصف', '-')).strip() if pd.notna(row.get('الوصف')) else "-"
            date_val = str(row.get('التاريخ', '-')).strip() if pd.notna(row.get('التاريخ')) else "-"
            
            if r not in region_events: region_events[r] = []
            if el not in element_orgs: element_orgs[el] = {}
            
            region_events[r].append({"director": director, "desc": desc, "org": org, "date": date_val, "city": city, "element": el, "id": row[id_col]})
            
        if len(df_expanded) > 0:
            subset_city = subset_regional + ['المدينة']
            df_unique_city = df_expanded.drop_duplicates(subset=subset_city)
            for _, row in df_unique_city.iterrows():
                r = row['cleaned_region']
                city = str(row.get('المدينة', '-')).strip()
                if r not in sub_charts: sub_charts[r] = {}
                sub_charts[r][city] = sub_charts[r].get(city, 0) + 1
        
        for _, row in df_unique_national.iterrows():
            el = str(row['العنصر']).strip() if pd.notna(row['العنصر']) else "-"
            org = str(row[org_col]).strip() if pd.notna(row[org_col]) else "جهة غير محددة"
            if el not in element_orgs: element_orgs[el] = {}
            element_orgs[el][org] = element_orgs[el].get(org, 0) + 1

        hierarchy_data = {}
        el_to_path = {}
        el_to_target = {}
        try:
            df_sheet2 = pd.read_excel(str(file_path), sheet_name="Sheet2")
            df_sheet2.columns = [str(col).strip() for col in df_sheet2.columns]
            
            el_col_s2 = [c for c in df_sheet2.columns if 'العنصر' in c or 'element' in str(c).lower()][0]
            path_col_s2 = [c for c in df_sheet2.columns if 'المسار' in c or 'path' in str(c).lower()][0]
            target_col_s2 = [c for c in df_sheet2.columns if 'target' in str(c).lower() or 'الهدف' in c or 'KPI' in str(c)]
            target_col_name = target_col_s2[0] if target_col_s2 else None
            
            df_sheet2[el_col_s2] = df_sheet2[el_col_s2].astype(str).str.strip()
            df_sheet2[path_col_s2] = df_sheet2[path_col_s2].astype(str).str.strip()
            
            el_to_path = dict(zip(df_sheet2[el_col_s2], df_sheet2[path_col_s2]))
            if target_col_name:
                el_to_target = dict(zip(df_sheet2[el_col_s2], pd.to_numeric(df_sheet2[target_col_name], errors='coerce').fillna(0)))
        except:
            pass
            
        elements_in_df = sorted([str(e).strip() for e in df_expanded['العنصر'].unique() if pd.notna(e)]) if 'العنصر' in df_expanded.columns else []
        all_possible_stage_elements = set(elements_in_df + list(el_to_path.keys()))
        unique_stage_elements = sorted([e for e in all_possible_stage_elements if e not in ['nan', '-', '']])
        for el in unique_stage_elements:
            base_el = el.replace(" - دولي", "")
            if el.endswith(" - دولي"):
                p_name = "المسار الدولي"
            else:
                p_name = el_to_path.get(el, el_to_path.get(base_el, "مسارات عامة"))
                if p_name == "nan" or p_name == "" or pd.isna(p_name): p_name = "مسارات عامة"
            
            c_val = int(raw_counts.get(el, 0))
            if p_name not in hierarchy_data: hierarchy_data[p_name] = {"total_path_count": 0, "elements": {}}
            hierarchy_data[p_name]["elements"][el] = c_val
            hierarchy_data[p_name]["total_path_count"] += c_val
            
            get_path_color(p_name)
            get_element_color(el)

            
        sorted_hierarchy = {str(k): hierarchy_data[k] for k in sorted(hierarchy_data.keys(), key=str)}
        for p in sorted_hierarchy: sorted_hierarchy[p]["elements"] = {str(k): sorted_hierarchy[p]["elements"][k] for k in sorted(sorted_hierarchy[p]["elements"].keys(), key=str)}
            
        return {
            "total": int(df_unique_national.shape[0]), "active_regions": sum(1 for v in counts_dict.values() if v > 0),
            "counts": counts_dict, "events": region_events, "sub_charts": sub_charts,
            "element_orgs": element_orgs, "hierarchy": sorted_hierarchy, "unique_elements": unique_stage_elements, "targets": el_to_target
        }
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f"❌ خطأ معالجة: {e}")
        return None

In [ ]:
data_execution = process_stage_data(EXCEL_PATH)
data_preparation = process_stage_data(EXCEL_PREP_PATH)
if data_preparation is None: data_preparation = data_execution.copy()

In [ ]:
element_images_prep = {el: [] for el in data_preparation.get("unique_elements", [])}
element_images_exec = {el: [] for el in data_execution.get("unique_elements", [])}
valid_extensions = ('.png', '.jpg', '.jpeg', '.webp', '.PNG', '.JPG', '.JPEG')

In [ ]:
import urllib.parse

In [ ]:
if ASSETS_DIR.exists():
    for root, dirs, files in os.walk(ASSETS_DIR):
        for file in files:
            if file.endswith(valid_extensions):
                rel_path = os.path.relpath(os.path.join(root, file), BASE_DIR)
                rel_path = rel_path.replace('\\', '/')
                path_parts = rel_path.split('/')
                safe_path = '/'.join([urllib.parse.quote(p) if p != 'assets' else p for p in path_parts])
                
                parent_dir = os.path.basename(root)
                grandparent_dir = os.path.basename(os.path.dirname(root)) if os.path.dirname(root) else ""
                
                matched_el = None
                for el in element_images_exec.keys():
                    base_el = el.replace(" - دولي", "")
                    if base_el == parent_dir or el == parent_dir:
                        matched_el = el
                        break
                    if base_el == grandparent_dir or el == grandparent_dir:
                        matched_el = el
                        break
                        
                if matched_el:
                    if not any(img['url'] == safe_path for img in element_images_exec[matched_el]):
                        element_images_exec[matched_el].append({
                            "url": safe_path,
                            "caption": parent_dir if parent_dir != matched_el and parent_dir != matched_el.replace(" - دولي", "") else "تصور أساسي"
                        })

In [ ]:
# Initialize variables to fix NameError
kpi_targets = data_preparation.get("targets", {}) if data_preparation else {}
if "المسار الدولي" not in path_colors_map:
    path_colors_map["المسار الدولي"] = "#8E44AD"

html_content = """<!DOCTYPE html>
<html dir="rtl" lang="ar">
<head>
    <meta charset="UTF-8"/>
    <title>لوحة القيادة الاستراتيجية والمقارنة المعيارية</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/chartjs-plugin-datalabels@2.0.0"></script>
    <link href="https://fonts.googleapis.com/css2?family=Tajawal:wght@400;500;700;800&display=swap" rel="stylesheet">
    <style>
        * { margin:0; padding:0; box-sizing: border-box; }
        body { font-family: 'Tajawal', sans-serif; background:#eef2f5; color:#2c3e50; display:flex; flex-direction:column; height:100vh; overflow:hidden; }
        
        #header { background:#1e3d59; color:white; padding:12px 25px; display:flex; justify-content:space-between; align-items:center; box-shadow:0 2px 8px rgba(0,0,0,0.15); z-index: 100; }
        #header h1 { font-size:18px; font-weight:800; }
        .stat-container { display:flex; gap:15px; }
        .stat-box { background:#17b978; padding:6px 15px; border-radius:6px; text-align:center; min-width:110px; border: 1px solid rgba(255,255,255,0.2); }
        .stat-box.orange-brand { background:#b85c38; } 
        .stat-val { font-size:17px; font-weight:800; }
        .stat-lbl { font-size:10px; opacity:0.9; font-weight:700; }
        
        #controls-wrapper { display: flex; justify-content: center; align-items: center; padding: 15px; background: #eef2f5; position: relative; z-index: 50; }
        .stage-switch-group { display: inline-flex; background: #fff; padding: 5px; border-radius: 30px; box-shadow: 0 2px 10px rgba(0,0,0,0.06); border: 1px solid #d1d9e0; gap: 5px; }
        .stage-btn { padding: 8px 24px; border: none; background: transparent; font-family: 'Tajawal'; font-weight: 800; font-size: 13px; color: #4a5568; border-radius: 25px; cursor: pointer; transition: all 0.3s ease; }
        .stage-btn.active-stage { background: #1e3d59; color: white; box-shadow: 0 3px 8px rgba(30,61,89,0.3); }
        .stage-btn.orange-mode.active-stage { background: #b85c38; color: white; box-shadow: 0 3px 8px rgba(184,92,56,0.3); }

        #main-wrapper { flex: 1; padding: 0 20px 20px 20px; overflow: hidden; display: flex; }
        .main-card { background: white; border-radius: 16px; box-shadow: 0 8px 25px rgba(0,0,0,0.06); flex: 1; display: flex; flex-direction: row-reverse; overflow: hidden; border: 1px solid #e1e8eb; transition: transform 0.35s cubic-bezier(0.4, 0, 0.2, 1), opacity 0.35s ease; }
        .slide-out-right { transform: translateX(50px); opacity: 0; }
        .slide-in-left { transform: translateX(-50px); opacity: 0; }
        
        #map-panel { flex:1.4; position:relative; background:#f8fafc; border-right: 2px solid #edf2f7; }
        svg { width:100%; height:100%; display: block; }
        .gov-path { stroke:#1e3d59; stroke-width:0.03; cursor:pointer; transition:all 0.2s ease; }
        .gov-path:hover { filter: brightness(0.8); stroke:#ff6f3c; stroke-width:0.12; }
        .gov-group.active-group .gov-path { stroke: #ff6f3c; stroke-width:0.18; filter: drop-shadow(0px 0px 4px rgba(0,0,0,0.3)); }

        
        #sidebar { flex:0.8; background:#fff; padding:20px; overflow-y:auto; min-width:340px; }
        .sec-title { font-size:15px; font-weight:800; color:#1e3d59; margin-bottom:15px; border-bottom:3px solid #ff6f3c; padding-bottom:6px; display:flex; justify-content:space-between; align-items:center; }
        
        .path-container { background:#f8fafc; border: 1px solid #e2e8f0; border-radius: 10px; margin-bottom: 12px; overflow: hidden; transition: all 0.2s; }
        .path-header { color: white; padding: 14px 16px; cursor: pointer; display: flex; justify-content: space-between; align-items: center; font-weight: 700; font-size: 13px; transition: filter 0.2s; }
        .path-header:hover { filter: brightness(1.15); }
        .path-container.active-path-filter { border: 3px solid #ff6f3c !important; transform: translateX(-3px); }
        .path-badge { background: rgba(255,255,255,0.25); color: white; padding: 2px 10px; border-radius: 12px; font-size: 11px; font-weight:800; border: 1px solid rgba(255,255,255,0.4); }
        .path-arrow { font-size: 11px; transition: transform 0.3s; }
        .path-container.open .path-arrow { transform: rotate(180deg); }
        .path-content { display: none; padding: 10px; background: #fff; }
        .path-container.open .path-content { display: block; }

        .element-row { margin-bottom:8px; font-size:12px; background:#fafafa; padding:12px; border-radius:8px; border:1px solid #e2e8f0; cursor:pointer; transition:all 0.2s ease; position:relative; }
        .element-row:hover { background:#f1f5f9; border-color:#cbd5e1; }
        .element-row.active-element-filter { background:#e8f5e9 !important; border-color:#17b978 !important; border-width:2px; transform: translateX(-3px); }
        .element-info { display:flex; justify-content:space-between; align-items:center; margin-bottom: 6px; }
        .element-name { font-weight:800; color:#2d3748; display:flex; align-items:center; gap:8px; }
        .element-badge { width:12px; height:12px; border-radius:3px; display:inline-block; }
        .element-count { font-weight:800; background:#fff; padding:2px 8px; border-radius:12px; font-size:11px; border:1px solid #cbd5e1; }
        
        .comp-progress-bg { width: 100%; background: #e2e8f0; height: 6px; border-radius: 3px; overflow: hidden; margin-top: 5px; display:none; }
        .comp-progress-fill { height: 100%; background: #17b978; border-radius: 3px; transition: width 0.5s; }

        #legend { position:absolute; bottom:20px; left:20px; background:rgba(255,255,255,0.95); padding:15px; border-radius:10px; box-shadow:0 4px 15px rgba(0,0,0,0.1); font-size:12px; z-index: 10; min-width:180px; border: 1px solid #e2e8f0; }
        .legend-title { font-weight:800; margin-bottom:10px; color:#1e3d59; font-size:13px; border-bottom: 1px solid #edf2f7; padding-bottom: 5px; }
        .legend-item { display:flex; align-items:center; gap:10px; margin-bottom:6px; font-weight:700; color:#4a5568;}
        .legend-color { width:18px; height:18px; border-radius:4px; border:1px solid rgba(0,0,0,0.1); }
        #tooltip { position:absolute; background:rgba(30,61,89,0.95); color:white; padding:12px 16px; border-radius:8px; font-size:12px; display:none; pointer-events:none; z-index:100; font-family:'Tajawal'; line-height:1.5; box-shadow: 0 4px 10px rgba(0,0,0,0.2); border: 1px solid rgba(255,255,255,0.1); }
        
        .big-modal-overlay { position:fixed; top:0; left:0; width:100%; height:100%; background:rgba(30,61,89,0.7); display:none; justify-content:center; align-items:center; z-index:2000; backdrop-filter: blur(2px); }
        .big-modal-window { background:#fff; width:90%; max-width:1100px; height:85vh; border-radius:16px; box-shadow:0 20px 50px rgba(0,0,0,0.4); overflow:hidden; display:flex; flex-direction:column; animation: modalFade 0.3s ease; }
        .big-modal-header { background:#1e3d59; color:white; padding:18px 25px; display:flex; justify-content:space-between; align-items:center; border-bottom: 4px solid #ff6f3c; }
        .big-modal-header h2 { font-size:16px; font-weight:800; }
        .close-big-modal { background:none; border:none; color:white; font-size:28px; cursor:pointer; font-weight:bold; transition: transform 0.2s; }
        .close-big-modal:hover { transform: scale(1.2); }
        .big-modal-body-layout { display:flex; flex:1; overflow:hidden; background:#fdfdfd; }
        .modal-right-col { flex:1; border-left:2px solid #edf2f7; padding:25px; overflow-y:auto; }
        .modal-left-col { flex:1.3; padding:25px; overflow-y:auto; background:#fafbfa; display:flex; flex-direction:column; }
        
        .city-progress-card { background:#fff; border:1px solid #e2e8f0; border-radius:10px; padding:14px 18px; margin-bottom:12px; cursor:pointer; transition:all 0.2s; box-shadow: 0 2px 5px rgba(0,0,0,0.02); }
        .city-progress-card:hover { border-color:#1e3d59; transform: translateY(-2px); box-shadow: 0 5px 10px rgba(0,0,0,0.05); }
        .city-meta-line { display:flex; justify-content:space-between; font-size:14px; font-weight:800; margin-bottom:8px; color: #2d3748; }
        .progress-bar-bg { background:#edf2f7; width:100%; height:8px; border-radius:4px; overflow:hidden; }
        .progress-bar-fill { height:100%; border-radius:4px; width:0%; transition: width 0.4s ease; }
        .modal-sec-title { font-size:14px; font-weight:800; color:#1e3d59; margin-bottom:15px; border-bottom:2px solid #ff6f3c; padding-bottom:5px; display:flex; justify-content:space-between; align-items:center; }
        
        .ev-card { background:#fff; border:1px solid #e2e8f0; border-radius:8px; padding:15px; margin-bottom:10px; box-shadow:0 2px 6px rgba(0,0,0,0.03); transition: all 0.2s; }
        .ev-header { display:flex; justify-content:space-between; margin-bottom:8px; font-size:11px; font-weight:800; }
        .ev-tag { color:white; padding:3px 10px; border-radius:15px; font-size:11px; }
        .ev-date { background:#e0f2fe; color:#0369a1; padding:3px 10px; border-radius:15px; border: 1px solid #bae6fd; }
        .ev-title { font-size:14px; font-weight:800; color:#1e293b; margin-bottom:6px; }
        .ev-desc { font-size:12px; color:#64748b; margin-bottom:8px; background:#f8fafc; padding:8px; border-radius:6px; border:1px dashed #cbd5e1; line-height: 1.5; }
        .ev-footer { display:flex; justify-content:space-between; font-size:11px; color:#94a3b8; border-top:1px solid #f1f5f9; padding-top:6px; font-weight: 700; }
        .view-orgs-btn { width: 100%; margin-top: 8px; padding: 6px; background: #f1f5f9; border: 1px solid #cbd5e1; border-radius: 5px; font-family: 'Tajawal'; font-size: 11px; font-weight: 700; color: #1e3d59; cursor: pointer; transition: all 0.2s; }
        .view-orgs-btn:hover { background: #1e3d59; color: white; }

        #comparison-panel { display: none; flex: 1; background: #f8fafc; padding: 25px; overflow-y: auto; }
        .comp-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap: 20px; margin-bottom: 25px; }
        .comp-card { background: white; padding: 20px; border-radius: 12px; border: 1px solid #e2e8f0; box-shadow: 0 4px 15px rgba(0,0,0,0.03); display: flex; flex-direction: column; }
        .comp-card-title { font-size: 14px; font-weight: 800; color: #1e3d59; margin-bottom: 10px; border-bottom: 2px solid #edf2f7; padding-bottom: 8px; }
        .comp-card-val { font-size: 32px; font-weight: 800; color: #b85c38; }
        .comp-card-sub { font-size: 12px; color: #64748b; font-weight: 700; margin-top: 5px; }
        .path-prog-container { margin-bottom: 15px; }
        .path-prog-title { display: flex; justify-content: space-between; font-size: 13px; font-weight: 800; color: #2d3748; margin-bottom: 5px; }
        .path-prog-bg { width: 100%; height: 10px; background: #e2e8f0; border-radius: 5px; overflow: hidden; }
        .path-prog-fill { height: 100%; border-radius: 5px; transition: width 0.8s ease; }
        .element-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(200px, 1fr)); gap: 15px; }
        .element-comp-card { background: #fff; border: 1px solid #cbd5e1; border-radius: 8px; padding: 15px; text-align: center; cursor: pointer; transition: all 0.2s; box-shadow: 0 2px 5px rgba(0,0,0,0.02); }
        .element-comp-card:hover { transform: translateY(-3px); box-shadow: 0 5px 15px rgba(0,0,0,0.05); border-color: #1e3d59; }
        .element-comp-name { font-size: 13px; font-weight: 800; color: #1e3d59; margin-bottom: 10px; height: 35px; }
        .element-comp-pct { font-size: 24px; font-weight: 800; margin-bottom: 5px; }

        /* Org Events Table */
        .org-events-table { width: 100%; border-collapse: collapse; font-size: 12px; background: #fff; border: 1px solid #e2e8f0; }
        .org-events-table th, .org-events-table td { border: 1px solid #e2e8f0; padding: 10px; text-align: right; }
        .org-events-table th { background: #1e3d59; color: white; font-weight: 800; position: sticky; top: 0; }
        .org-events-table tr:nth-child(even) { background: #f8fafc; }
        .img-filter-btn { background: #b85c38; color: white; border: none; padding: 4px 8px; border-radius: 4px; cursor: pointer; font-size: 10px; font-weight: 800; font-family: 'Tajawal'; }
        .img-filter-btn:hover { background: #1e3d59; }

        /* Lightbox Gallery Styles */
        #lightbox-overlay { position: fixed; top: 0; left: 0; width: 100%; height: 100%; background: rgba(15,23,42,0.95); z-index: 3000; display: none; justify-content: center; align-items: center; flex-direction: column; backdrop-filter: blur(5px); }
        #lightbox-img { max-width: 90%; max-height: 80vh; border-radius: 8px; box-shadow: 0 10px 30px rgba(0,0,0,0.5); object-fit: contain; }
        #lightbox-caption { color: white; font-size: 16px; font-weight: 800; margin-top: 15px; }
        .lightbox-nav { position: absolute; top: 50%; transform: translateY(-50%); background: rgba(255,255,255,0.1); color: white; border: none; font-size: 36px; padding: 15px 20px; cursor: pointer; border-radius: 50%; transition: background 0.3s; z-index: 3010; }
        .lightbox-nav:hover { background: rgba(255,255,255,0.3); }
        #lightbox-prev { left: 30px; }
        #lightbox-next { right: 30px; }
        #lightbox-close { position: absolute; top: 30px; right: 40px; background: none; border: none; color: white; font-size: 40px; cursor: pointer; transition: transform 0.2s; z-index: 3010; }
        #lightbox-close:hover { transform: scale(1.2); }
        @keyframes modalFade { from { opacity:0; transform:scale(0.95) translateY(10px); } to { opacity:1; transform:scale(1) translateY(0); } }
    </style>
</head>
<body>
    <div id="header">
        <h1>لوحة القيادة الاستراتيجية للمبادرات</h1>
        <div class="stat-container">
            <div class="stat-box orange-brand"><div class="stat-val" id="top-total-counter">0</div><div class="stat-lbl" id="top-total-label">مستهدفات المرحلة</div></div>
            <div class="stat-box"><div class="stat-val" id="top-active-regions">0</div><div class="stat-lbl">المناطق المغطاة</div></div>
        </div>
    </div>

    
    <div id="controls-wrapper">
        <div class="stage-switch-group">
            <button class="stage-btn active-stage" id="btn-prep" onclick="triggerStageChange('preparation')">🔄 مرحلة الاعتماد والإعداد</button>
            <button class="stage-btn" id="btn-exec" onclick="triggerStageChange('execution')">🚀 مرحلة التنفيذ والمخرجات</button>
            <button class="stage-btn orange-mode" id="btn-comp" onclick="triggerStageChange('comparison')">⚖️ مقارنة نسبة الإنجاز</button>
        </div>
    </div>

    <div id="main-wrapper">
        <div class="main-card" id="main-card">
                        <div id="map-panel">
                <svg id="ksa-svg" viewBox="34.0 16.0 22.0 17.0"></svg>
                <div id="legend">
                    <div class="legend-title" id="legend-main-title">📊 توزيع مخرجات المبادرات:</div>
                    <div class="legend-item"><div class="legend-color" id="leg-c0"></div> <span id="leg-t0">0</span></div>
                    <div class="legend-item"><div class="legend-color" id="leg-c1"></div> <span id="leg-t1">-</span></div>
                    <div class="legend-item"><div class="legend-color" id="leg-c2"></div> <span id="leg-t2">-</span></div>
                    <div class="legend-item"><div class="legend-color" id="leg-c3"></div> <span id="leg-t3">-</span></div>
                    <div class="legend-item"><div class="legend-color" id="leg-c4"></div> <span id="leg-t4">-</span></div>
                </div>
                                <div id="tooltip"></div>
            </div>
            
                        <div id="sidebar" style="flex:1; padding-right:15px; overflow-y:auto; display:flex; flex-direction:column; gap:15px;">
                <div class="sec-title" id="sidebar-stage-title">🎯 مسارات المبادرات</div>
                <div id="national-elements-list" style="margin-bottom:15px;"></div>
            </div>

            <div id="comparison-panel"></div>
        </div>
    </div>
    
    <div id="big-dashboard-modal" class="big-modal-overlay" onclick="closeBigModal()">
        <div class="big-modal-window" onclick="event.stopPropagation()">
            <div class="big-modal-header"><h2 id="big-modal-title">📍 المؤشرات التفصيلية للمنطقة</h2><button class="close-big-modal" onclick="closeBigModal()">&times;</button></div>
            <div class="big-modal-body-layout">
                <div class="modal-right-col"><div class="modal-sec-title">📈 كثافة التوزيع حسب المحافظات والمدن</div><div id="modal-cities-progress-list"></div></div>
                <div class="modal-left-col">
                    <div class="modal-sec-title"><span id="modal-timeline-title">📅 السجل الزمني التنفيذي للمخرجات</span><button class="stage-btn" id="clear-city-filter-btn" style="display:none; padding:4px 10px; margin:0;" onclick="clearCityFilter()">إلغاء تصفية المدينة</button></div>
                    <div id="modal-events-timeline" style="overflow-y:auto; flex:1; padding-right:5px;"></div>
                </div>
            </div>
        </div>
    </div>

    <div id="element-details-modal" class="big-modal-overlay" onclick="closeElementModal()">
        <div class="big-modal-window" onclick="event.stopPropagation()">
            <div class="big-modal-header"><h2 id="element-modal-title">🔍 تفاصيل العنصر</h2><button class="close-big-modal" onclick="closeElementModal()">&times;</button></div>
            <div class="big-modal-body-layout">
                <div class="modal-right-col" style="flex:0.8;">
                    <div class="modal-sec-title">🏢 الجهات المشاركة بالدولة</div>
                    <div id="element-orgs-list"></div>
                </div>
                <div class="modal-left-col" style="flex:1.5;">
                    
                                        
                    <div class="modal-sec-title">📊 المعتمد مقابل التنفيذ والمتبقي</div>
                    <div id="element-kpi-view"></div>
                    
                    <div class="modal-sec-title" id="modal-images-title" style="margin-top:20px;">📸 التصورات البصرية</div>
                    <div id="element-images-view" style="display:flex; flex-wrap:wrap; gap:10px; margin-top:10px; margin-bottom:20px;"></div>


                    
                    <div class="modal-sec-title" id="org-events-title" style="margin-top:20px; display:none;">📋 مخرجات الجهة المختارة</div>
                    <div id="org-events-table-container" style="display:none; margin-bottom: 20px; max-height: 250px; overflow-y:auto;"></div>
                    
                    
                </div>
            </div>
        </div>
    </div>

    <div id="lightbox-overlay" onclick="closeLightbox()">
        <button id="lightbox-close" onclick="closeLightbox()">&times;</button>
        <button class="lightbox-nav" id="lightbox-next" onclick="changeLightboxImage(1, event)">&#10095;</button>
        <img id="lightbox-img" src="" alt="">
        <div id="lightbox-caption"></div>
        <button class="lightbox-nav" id="lightbox-prev" onclick="changeLightboxImage(-1, event)">&#10094;</button>
    </div>
"""
html_script = """
    <script>
        const STAGES_DATA = { preparation: __PREP_DATA_JSON__, execution: __EXEC_DATA_JSON__ };
        const KPI_TARGETS = __KPI_TARGETS_JSON__;
        const ELEMENT_IMAGES_PREP = __ELEMENT_IMAGES_PREP_JSON__;
        const ELEMENT_IMAGES_EXEC = __ELEMENT_IMAGES_EXEC_JSON__;
        const REGION_AR = __REGIONS_MAP_AR__; const PATH_COLORS = __PATH_COLORS_JSON__; const ELEMENT_COLORS = __ELEMENT_COLORS_JSON__;
        
        let currentStage = 'preparation'; let currentOpenRegionKey = null; 
        let selectedElementFilter = null; let selectedPathFilter = null; 
        let currentGalleryImages = []; let activeLightboxImages = []; let currentGalleryIndex = 0;
        
        const tip = document.getElementById('tooltip'); const card = document.getElementById('main-card');
        const DEFAULT_SCALES = ["#f7fbff", "#c6dbef", "#6baed6", "#2171b5", "#08306b"];
        const COMP_SCALES = ["#ffebee", "#ffcc80", "#fff59d", "#a5d6a7", "#4caf50"]; 

        function generateChoroplethScales(baseHex) { let hex = baseHex.replace('#', ''); if(hex.length === 3) hex = hex.split('').map(c => c+c).join(''); let r = parseInt(hex.substring(0,2), 16); let g = parseInt(hex.substring(2,4), 16); let b = parseInt(hex.substring(4,6), 16); return [ "#f8fafc", `rgba(${r}, ${g}, ${b}, 0.3)`, `rgba(${r}, ${g}, ${b}, 0.6)`, `rgba(${r}, ${g}, ${b}, 0.85)`, `rgba(${r}, ${g}, ${b}, 1.0)` ]; }
        function updateLegendUI(title, scales, breakPoints, isComp = false) { 
            document.getElementById("legend-main-title").textContent = title; 
            for(let i=0; i<5; i++) document.getElementById(`leg-c${i}`).style.background = scales[i]; 
            if(isComp) {
                document.getElementById("leg-t0").textContent = `0% (لم يبدأ)`; document.getElementById("leg-t1").textContent = `1% - 25%`; document.getElementById("leg-t2").textContent = `26% - 50%`; document.getElementById("leg-t3").textContent = `51% - 75%`; document.getElementById("leg-t4").textContent = `76% - 100%+`;
            } else {
                document.getElementById("leg-t0").textContent = `0 مخرج`; document.getElementById("leg-t1").textContent = `1 - ${breakPoints[0]} مخرج`; document.getElementById("leg-t2").textContent = `${breakPoints[0]+1} - ${breakPoints[1]} مخرج`; document.getElementById("leg-t3").textContent = `${breakPoints[1]+1} - ${breakPoints[2]} مخرج`; document.getElementById("leg-t4").textContent = `أكثر من ${breakPoints[2]} مخرج`;
            }
        }

        function triggerStageChange(stageKey) {
            if(currentStage === stageKey) return;
            card.className = "main-card slide-out-right";
            setTimeout(() => {
                currentStage = stageKey;
                selectedElementFilter = null; selectedPathFilter = null;
                document.querySelectorAll('.stage-btn').forEach(btn => btn.classList.remove('active-stage'));
                if(stageKey === 'preparation') document.getElementById('btn-prep').classList.add('active-stage');
                else if(stageKey === 'execution') document.getElementById('btn-exec').classList.add('active-stage');
                else if(stageKey === 'comparison') document.getElementById('btn-comp').classList.add('active-stage');
                
                const mapPanel = document.getElementById('map-panel');
                const sidebar = document.getElementById('sidebar');
                const compPanel = document.getElementById('comparison-panel');
                
                if(stageKey === 'comparison') {
                    mapPanel.style.display = 'none';
                    sidebar.style.display = 'none';
                    compPanel.style.display = 'block';
                    
                    let totalExec = 0; let totalTarget = 0;
                    Object.keys(STAGES_DATA.execution.hierarchy).forEach(p => Object.keys(STAGES_DATA.execution.hierarchy[p].elements).forEach(el => totalExec += STAGES_DATA.execution.hierarchy[p].elements[el]));
                    Object.keys(KPI_TARGETS).forEach(el => totalTarget += KPI_TARGETS[el]);
                    const pctTotal = totalTarget === 0 ? (totalExec>0?100:0) : Math.round((totalExec / totalTarget) * 100);
                    
                    document.getElementById('top-total-counter').textContent = pctTotal + "%";
                    document.getElementById('top-total-label').textContent = "مؤشر الأداء الشامل (من الهدف)";
                    document.getElementById('top-active-regions').textContent = totalExec;
                    document.getElementById('top-active-regions').nextElementSibling.textContent = "إجمالي المنفذ مقابل الأهداف";
                    
                    renderComparisonDashboard(totalExec, totalTarget, pctTotal);
                } else {
                    mapPanel.style.display = 'block';
                    sidebar.style.display = 'block';
                    compPanel.style.display = 'none';
                    
                    const sInfo = STAGES_DATA[currentStage];
                    let tCount = sInfo.total;
                    if(stageKey === 'preparation') {
                        tCount = 0;
                        Object.keys(KPI_TARGETS).forEach(el => tCount += KPI_TARGETS[el]);
                    }
                    document.getElementById('top-total-counter').textContent = tCount;
                    document.getElementById('top-active-regions').textContent = sInfo.active_regions;
                    document.getElementById('top-active-regions').nextElementSibling.textContent = "المناطق المغطاة";
                    document.getElementById('top-total-label').textContent = stageKey === 'preparation' ? "إجمالي الأهداف (KPI)" : "إجمالي المخرجات المنفذة";
                    document.getElementById('sidebar-stage-title').textContent = stageKey === 'preparation' ? "🎯 مسارات الاعتماد والأهداف" : "🚀 مسارات التنفيذ والمخرجات";
                    
                    renderHierarchySidebar();
                    refreshMapColors(); 
                }
                card.className = "main-card slide-in-left";
                void card.offsetWidth; 
                card.className = "main-card";
            }, 350);
        }
       
        function renderComparisonDashboard(totalExec, totalTarget, pctTotal) {
            const compPanel = document.getElementById('comparison-panel');
            let html = `
                <div class="sec-title" style="font-size:20px; border-bottom:4px solid #17b978;">📊 لوحة قياس مؤشرات الأداء (KPIs)</div>
                <div class="comp-grid">
                    <div class="comp-card">
                        <div class="comp-card-title">مؤشر الإنجاز الوطني</div>
                        <div class="comp-card-val" style="color: ${pctTotal >= 100 ? '#17b978' : '#f57c00'}">${pctTotal}%</div>
                        <div class="comp-card-sub">إجمالي المنفذ مقابل المستهدف</div>
                    </div>
                    <div class="comp-card">
                        <div class="comp-card-title">إجمالي المخرجات المنفذة</div>
                        <div class="comp-card-val" style="color: #1e3d59">${totalExec}</div>
                        <div class="comp-card-sub">مخرج تم تسجيله</div>
                    </div>
                    <div class="comp-card">
                        <div class="comp-card-title">المستهدف العام (KPI)</div>
                        <div class="comp-card-val" style="color: #b85c38">${totalTarget}</div>
                        <div class="comp-card-sub">الهدف المراد تحقيقه</div>
                    </div>
                </div>
            `;
            html += `<div class="sec-title">🎯 تفصيل مؤشرات الأداء حسب المسار</div>`;
            const hierarchy = STAGES_DATA.execution.hierarchy;
            
            let pathChartConfigs = [];
            
            Object.keys(hierarchy).forEach((pName, index) => {
                let pExec = 0; let pTarget = 0;
                let elLabels = []; let elData = []; let elColors = [];
                
                // Calculate total covered cities for this path
                let pathCities = new Set();
                Object.keys(hierarchy[pName].elements).forEach(el => {
                    let e_exec = hierarchy[pName].elements[el] || 0;
                    pExec += e_exec;
                    pTarget += KPI_TARGETS[el] || 0;
                    
                    let elBase = el.replace(' - دولي', '');
                    const colorEl = ELEMENT_COLORS[el] || ELEMENT_COLORS[elBase] || PATH_COLORS[pName] || "#1e3d59";
                    
                    if(e_exec > 0) {
                        elLabels.push(el);
                        elData.push(e_exec);
                        elColors.push(colorEl);
                    }
                    
                    // Count unique cities
                    Object.keys(STAGES_DATA.execution.events).forEach(reg => {
                        STAGES_DATA.execution.events[reg].forEach(ev => {
                            if (ev.element === el) pathCities.add(ev.city);
                        });
                    });
                });
                
                let coveredCitiesCount = pathCities.size;
                
                const pct = pTarget === 0 ? (pExec>0?100:0) : Math.min(Math.round((pExec / pTarget) * 100), 100);
                const color = pct >= 100 ? '#17b978' : (pct >= 50 ? '#fbc02d' : '#e64a19');
                
                const pColor = PATH_COLORS[pName] || '#1e3d59';
                const chartId = `chart_path_${index}`;
                const containerId = `path_container_${index}`;
                pathChartConfigs.push({id: chartId, labels: elLabels, data: elData, colors: elColors});

                html += `
                <div style="background:#fff; border-radius:12px; border:1px solid #e2e8f0; margin-bottom:25px; overflow:hidden;">
                    <div class="path-header" style="background:${pColor}; color:white; padding:15px; cursor:pointer;" onclick="toggleCompPath('${containerId}')">
                        
                                                <div style="display:flex; justify-content:space-between; align-items:center; width: 100%;">
                            <span style="font-size:16px; font-weight:800;">📁 ${pName}</span>
                            <span style="font-size:14px; background:rgba(255,255,255,0.2); padding:4px 12px; border-radius:12px;">نظرة عامة ▼</span>
                        </div>

                    </div>
                    
                    <div id="${containerId}" style="display:none; padding:20px; background:#f8fafc;">
                        <div style="display:flex; flex-direction:row-reverse; flex-wrap:wrap; gap:20px; margin-bottom:20px; background:#fff; padding:15px; border-radius:8px; border:1px solid #e2e8f0;">
                            
                            <div style="flex:1; min-width:250px; display:flex; flex-direction:column; align-items:center; justify-content:center; border-left:1px solid #edf2f7; padding-left:15px;">
                                <div style="font-size:14px; font-weight:800; color:#1e3d59; margin-bottom:10px; text-align:center;">نسبة المخرجات للعناصر في المسار</div>
                                <div style="width:220px; height:220px;"><canvas id="${chartId}"></canvas></div>
                            </div>
                            
                            <div style="flex:1.5; min-width:300px; display:flex; flex-direction:column; justify-content:center; gap:15px;">
                                <div style="display:flex; gap:10px;">
                                    <div style="flex:1; background:#e0f2fe; padding:15px; border-radius:8px; border:1px solid #bae6fd; text-align:center;">
                                        <div style="font-size:24px; font-weight:800; color:#0369a1;">${coveredCitiesCount}</div>
                                        <div style="font-size:12px; font-weight:700; color:#0c4a6e;">المدن المغطاة</div>
                                    </div>
                                    <div style="flex:1; background:#fef3c7; padding:15px; border-radius:8px; border:1px solid #fde68a; text-align:center;">
                                        <div style="font-size:24px; font-weight:800; color:#b45309;">${pTarget}</div>
                                        <div style="font-size:12px; font-weight:700; color:#78350f;">إجمالي المعتمد</div>
                                    </div>
                                    <div style="flex:1; background:#dcfce7; padding:15px; border-radius:8px; border:1px solid #bbf7d0; text-align:center;">
                                        <div style="font-size:24px; font-weight:800; color:#15803d;">${pExec}</div>
                                        <div style="font-size:12px; font-weight:700; color:#14532d;">إجمالي المنفذ</div>
                                    </div>
                                </div>
                                
                                <div class="path-prog-container" style="margin-top:10px;">
                                    <div class="path-prog-title"><span>نسبة إنجاز المسار ككل</span> <span>${pct}%</span></div>
                                    <div class="path-prog-bg" style="height:14px; border-radius:7px;"><div class="path-prog-fill" style="width: ${pct}%; background: ${color}; border-radius:7px;"></div></div>
                                </div>
                            </div>
                            
                        </div>
                        
                        <div class="sec-title" style="font-size:14px; margin-top:20px; border-bottom:2px solid #cbd5e1;">تفصيل العناصر</div>
                        <div class="element-grid" style="grid-template-columns: repeat(auto-fill, minmax(220px, 1fr));">
                `;
                
                Object.keys(hierarchy[pName].elements).forEach(el => {
                    const target = KPI_TARGETS[el] || 0;
                    const exec = hierarchy[pName].elements[el] || 0;
                    const pctEl = target === 0 ? (exec>0?100:0) : Math.round((exec / target) * 100);
                    const progColorEl = pctEl >= 100 ? '#17b978' : (pctEl >= 50 ? '#fbc02d' : '#e64a19');
                    
                    let elBase = el.replace(' - دولي', '');
                    const colorEl = ELEMENT_COLORS[el] || ELEMENT_COLORS[elBase] || PATH_COLORS[pName] || "#1e3d59";
                    
                    html += `
                            <div class="element-comp-card" onclick="openElementDetails('${el}', event)">
                                <div class="element-comp-name" style="display:flex; align-items:center; gap:8px;">
                                    <span style="display:inline-block; width:12px; height:12px; border-radius:50%; background:${colorEl};"></span>
                                    <span>${el}</span>
                                </div>
                                <div class="element-comp-pct" style="color: ${progColorEl}">${pctEl}%</div>
                                <div style="font-size:12px; font-weight:700; color:#64748b; margin-bottom:10px;">🎯 هدف: ${target} | 🚀 منفذ: ${exec}</div>
                                <div class="progress-bar-bg" style="height:6px; margin-bottom:10px;"><div class="progress-bar-fill" style="width:${pctEl}%; background:${progColorEl};"></div></div>
                                <button class="view-orgs-btn">تحليل تفصيلي للجهات</button>
                            </div>
                    `;
                });
                
                html += `
                        </div>
                    </div>
                </div>`;
            });
            
            compPanel.innerHTML = html;
            
            // Add script to window for toggling
            if(!window.toggleCompPath) {
                window.toggleCompPath = function(id) {
                    const el = document.getElementById(id);
                    if(el.style.display === 'none') {
                        el.style.display = 'block';
                    } else {
                        el.style.display = 'none';
                    }
                };
            }
            
            
            // Register datalabels
            Chart.register(ChartDataLabels);
            
            // Initialize charts

            setTimeout(() => {
                pathChartConfigs.forEach(cfg => {
                    if (cfg.data.length > 0) {
                        const ctx = document.getElementById(cfg.id);
                        if(ctx) {
                            
                new Chart(ctx, {
                    type: 'doughnut',
                    data: { labels: cfg.labels, datasets: [{ data: cfg.data, backgroundColor: cfg.colors, borderWidth:0 }] },
                    options: { 
                        responsive: true, maintainAspectRatio: false, 
                        plugins: { 
                            legend: { display: false },
                            tooltip: { bodyFont: {family:'Tajawal'}, titleFont: {family:'Tajawal'} },
                            datalabels: {
                                color: '#fff',
                                font: { family: 'Tajawal', weight: 'bold', size: 10 },
                                formatter: (value, ctx) => {
                                    let sum = 0;
                                    let dataArr = ctx.chart.data.datasets[0].data;
                                    dataArr.map(data => { sum += data; });
                                    let percentage = (value*100 / sum).toFixed(1)+"%";
                                    if(value === 0) return null;
                                    let label = ctx.chart.data.labels[ctx.dataIndex];
                                    let shortLabel = label.split(' ')[0] + '...';
                                    if (label.length < 10) shortLabel = label;
                                    return shortLabel + '\\n' + percentage;
                                },
                                anchor: 'center',
                                align: 'center'
                            }
                        },
                        cutout: '65%'
                    }
                });

                        }
                    } else {
                        const ctx = document.getElementById(cfg.id);
                        if(ctx) {
                             ctx.parentElement.innerHTML = '<div style="color:#94a3b8; font-size:12px; text-align:center; padding-top:80px;">لا توجد مخرجات منفذة</div>';
                        }
                    }
                });
            }, 100);
        }

        function renderHierarchySidebar() {
            const container = document.getElementById("national-elements-list");
            if (!container) return; 
            container.innerHTML = "";
            const baseHierarchy = STAGES_DATA.preparation.hierarchy; 
            const currentH = currentStage === 'comparison' ? baseHierarchy : STAGES_DATA[currentStage].hierarchy;
            Object.keys(currentH).forEach(pathName => {
                const pathObj = currentH[pathName];
                const pathColor = PATH_COLORS[pathName] || "#1e3d59";
                const boxId = "path-box-" + btoa(unescape(encodeURIComponent(pathName))).replace(/=/g, '');
                let pBadgeTxt = `${pathObj.total_path_count} مخرج`;
                if(currentStage === 'comparison') {
                    let execP = 0; let targetP = 0;
                    Object.keys(pathObj.elements).forEach(el => { 
                        execP += (STAGES_DATA.execution.hierarchy[pathName]?.elements[el] || 0); 
                        targetP += (KPI_TARGETS[el] || 0);
                    });
                    pBadgeTxt = `مُنفذ: ${execP} من هدف ${targetP}`;
                } else if(currentStage === 'preparation') {
                    let targetP = 0;
                    Object.keys(pathObj.elements).forEach(el => { targetP += (KPI_TARGETS[el] || 0); });
                    pBadgeTxt = `معتمد: ${targetP}`;
                } else if(currentStage === 'execution') {
                    pBadgeTxt = `مُنفذ: ${pathObj.total_path_count} مخرج`;
                }
                const pathDiv = document.createElement("div"); 
                pathDiv.className = "path-container"; pathDiv.id = boxId;
                pathDiv.innerHTML = `
                    <div class="path-header" style="background:${pathColor}" onclick="handlePathHeaderClick('${pathName}')">
                        <span>📁 ${pathName}</span>
                        <div style="display:flex; align-items:center; gap:8px;">
                            <span class="path-badge">${pBadgeTxt}</span>
                            <span class="path-arrow">▼</span>
                        </div>
                    </div>
                    <div class="path-content"></div>
                `;
                const contentDiv = pathDiv.querySelector('.path-content');
                Object.keys(pathObj.elements).forEach(el => {
                    const elColor = ELEMENT_COLORS[el] || pathColor;
                    const targetKPI = KPI_TARGETS[el] || 0;
                    const actualExec = STAGES_DATA.execution.hierarchy[pathName]?.elements[el] || 0;
                    const actualPrep = STAGES_DATA.preparation.hierarchy[pathName]?.elements[el] || 0;
                    let elCountHtml = ''; let compBar = '';
                    if(currentStage === 'comparison') {
                        const pct = targetKPI === 0 ? (actualExec>0?100:0) : Math.min(Math.round((actualExec/targetKPI)*100), 100);
                        elCountHtml = `<span class="element-count" style="color:#1e3d59; border-color:#cbd5e1">${actualExec} / ${targetKPI} (${pct}%)</span>`;
                        compBar = `<div class="comp-progress-bg" style="display:block;"><div class="comp-progress-fill" style="width:${pct}%; background:#17b978"></div></div>`;
                    } else if(currentStage === 'preparation') {
                        elCountHtml = `<span class="element-count" style="color: ${elColor}; border-color: ${elColor}">المعتمد: ${targetKPI}</span>`;
                    } else if(currentStage === 'execution') {
                        elCountHtml = `<span class="element-count" style="color: ${elColor}; border-color: ${elColor}">المنفذ: ${actualExec} مخرج</span>`;
                    }
                    const elRow = document.createElement("div");
                    elRow.className = "element-row";
                    elRow.id = "el-row-" + btoa(unescape(encodeURIComponent(el))).replace(/=/g, '');
                    elRow.innerHTML = `
                        <div class="element-info" onclick="handleElementRowClick('${el}', event)">
                            <span class="element-name"><span class="element-badge" style="background: ${elColor}"></span>${el}</span>
                            ${elCountHtml}
                        </div>
                        ${compBar}
                        <button class="view-orgs-btn" onclick="openElementDetails('${el}', event)">🔍 التفاصيل ومؤشرات الأداء</button>
                    `;
                    contentDiv.appendChild(elRow);
                });
                container.appendChild(pathDiv);
            });
        }
        function openElementDetails(elName, event) {
            event.stopPropagation();
            document.getElementById('element-details-modal').style.display = 'flex';
            document.getElementById('element-modal-title').textContent = "🔍 تفاصيل العنصر: " + elName;
            
            document.getElementById('org-events-table-container').style.display = 'none';
            document.getElementById('org-events-title').style.display = 'none';
            
            const isPrep = (currentStage === 'preparation');
            const images = isPrep ? ELEMENT_IMAGES_PREP[elName] : ELEMENT_IMAGES_EXEC[elName];
            const stageInfo = STAGES_DATA[currentStage === 'comparison' ? 'execution' : currentStage];
            const orgs = stageInfo.element_orgs[elName] || {};
            
            let orgsHtml = "";
            const sortedOrgs = Object.entries(orgs).sort((a,b)=>b[1]-a[1]);
            const maxOrgCount = sortedOrgs.length > 0 ? sortedOrgs[0][1] : 1;
            
            sortedOrgs.forEach(([org, cnt]) => {
                const percentage = (cnt / maxOrgCount) * 100;
                orgsHtml += `
                    <div class="city-progress-card org-card-item" onclick="renderOrgEvents('${org}', '${elName}')" style="padding: 10px 15px; border-left: 4px solid #b85c38;">
                        <div class="city-meta-line" style="font-size: 13px;">
                            <span>🏢 ${org}</span>
                            <span style="color:#b85c38;">${cnt} مساهمة</span>
                        </div>
                        <div class="progress-bar-bg" style="height: 6px;">
                            <div class="progress-bar-fill" style="width: ${percentage}%; background: #1e3d59;"></div>
                        </div>
                        <div style="font-size:10px; color:#64748b; margin-top:4px;">اضغط لعرض المخرجات والصور</div>
                    </div>
                `;
            });
            document.getElementById('element-orgs-list').innerHTML = orgsHtml || "<div style='text-align:center; padding:10px; font-weight:bold; color:#94a3b8;'>لا توجد جهات مسجلة لهذا العنصر</div>";
            
            const targetKPI = KPI_TARGETS[elName] || 0;
            let execCount = 0;
            Object.values(STAGES_DATA.execution.hierarchy).forEach(p => { if(p.elements[elName]) execCount = p.elements[elName]; });
            
            let missingCount = Math.max(0, targetKPI - execCount);
            let pct = targetKPI === 0 ? (execCount>0?100:0) : Math.round((execCount/targetKPI)*100);
            
            document.getElementById('element-kpi-view').innerHTML = `
                <div style="background:#f8fafc; padding:15px; border-radius:10px; border:1px solid #e2e8f0; text-align:center;">
                    <div style="font-size:32px; font-weight:800; color:${pct>=100?'#17b978':(pct>50?'#fbc02d':'#e64a19')}">${pct}%</div>
                    <div style="margin-top:10px; display:flex; justify-content:space-around; font-size:13px; font-weight:800; background:#fff; padding:10px; border-radius:8px; border:1px solid #e2e8f0;">
                        <div>🎯 معتمد: <span style="color:#1e3d59; font-size:15px;">${targetKPI}</span></div>
                        <div>🚀 منفذ: <span style="color:#17b978; font-size:15px;">${execCount}</span></div>
                        <div>⚠️ متبقي: <span style="color:#e64a19; font-size:15px;">${missingCount}</span></div>
                    </div>
                </div>
            `;
            
            
            // Render gallery inside modal

            // Render gallery inside modal
            const elImages = currentStage === 'preparation' ? [] : ELEMENT_IMAGES_EXEC[elName];
            renderGallery(elImages || []);


        }
    
        function renderOrgEvents(orgName, elName) {
            const tableContainer = document.getElementById('org-events-table-container');
            
            if (currentStage !== 'comparison') {
                document.getElementById('org-events-title').style.display = 'block';
                tableContainer.style.display = 'block';
                tableContainer.innerHTML = `<div style="text-align:center; padding:15px; background:#fff; border:1px solid #e2e8f0; border-radius:8px; font-weight:bold; color:#64748b;">جدول المخرجات متاح فقط في شاشة مقارنة نسبة الإنجاز</div>`;
                return;
            }
            
            document.getElementById('org-events-title').style.display = 'block';
            tableContainer.style.display = 'block';
            
            const evData = STAGES_DATA.execution.events;
            let matchingEventsMap = {};
            
            Object.keys(evData).forEach(reg => {
                evData[reg].forEach(e => {
                    if (e.element === elName && e.org === orgName) {
                        const uniqueKey = e.id + "_" + e.director;
                        if (!matchingEventsMap[uniqueKey]) {
                            matchingEventsMap[uniqueKey] = {
                                director: e.director, desc: e.desc, date: e.date, cities: new Set([e.city])
                            };
                        } else {
                            matchingEventsMap[uniqueKey].cities.add(e.city);
                        }
                    }
                });
            });
            
            const rows = Object.values(matchingEventsMap);
            if(rows.length === 0) {
                tableContainer.innerHTML = `<div style="text-align:center; padding:15px; background:#fff; border:1px solid #e2e8f0; border-radius:8px;">لا توجد تفاصيل متاحة لهذه الجهة</div>`;
                return;
            }
            
            let tableHtml = `<table class="org-events-table">
                <thead><tr><th>اسم المخرج</th><th>الوصف</th><th>التاريخ</th><th>المدن المغطاة</th><th>صور</th></tr></thead><tbody>`;
                
            rows.forEach(r => {
                const citiesStr = Array.from(r.cities).join('، ');
                // Check if any images exist for this director
                const hasImages = currentGalleryImages && currentGalleryImages.length > 0;
                let imgBtnHtml = hasImages ? `<button class="img-filter-btn" onclick="filterGalleryByDirector('${r.director}')">عرض صوره</button>` : `<span style="color:#94a3b8; font-size:10px;">لا يوجد</span>`;
                
                tableHtml += `<tr>
                    <td style="font-weight:700; color:#1e3d59;">${r.director}</td>
                    <td>${r.desc}</td>
                    <td>${r.date}</td>
                    <td style="font-size:10px;">${citiesStr}</td>
                    <td style="text-align:center;">${imgBtnHtml}</td>
                </tr>`;
            });
            tableHtml += `</tbody></table>`;
            tableContainer.innerHTML = tableHtml;
        }

                        function renderGallery(images) {
            const imgContainer = document.getElementById('element-images-view');
            const imgTitle = document.getElementById('modal-images-title');
            
            if (currentStage === 'comparison') {
                imgContainer.style.display = 'none';
                imgTitle.style.display = 'none';
                currentGalleryImages = images || [];
                return;
            }
            
            imgContainer.style.display = 'flex';
            imgTitle.style.display = 'block';
            
            if(!images || images.length === 0) {
                imgContainer.innerHTML = `<div class="no-assets-msg" style="width:100%; text-align:center; padding:20px; background:#fff; border:1px dashed #cbd5e1; border-radius:8px;">لا توجد تصورات بصرية مضافة</div>`;
                currentGalleryImages = [];
            } else {
                currentGalleryImages = images;
                imgContainer.innerHTML = images.map((img, idx) => `
                    <div style="width:48%; border:1px solid #e2e8f0; border-radius:8px; overflow:hidden; background:#fff; cursor:pointer; transition: transform 0.2s;" onclick="openLightbox(${idx})" onmouseover="this.style.transform='scale(1.03)'" onmouseout="this.style.transform='scale(1)'">
                        <img src="${img.url}" style="width:100%; height:180px; object-fit:cover;" alt="${img.caption}">
                        <div style="padding:8px; font-size:11px; text-align:center; font-weight:700;">${img.caption}</div>
                    </div>`).join('');
            }
        }
        
                        function filterGalleryByDirector(directorName) {
            if(!currentGalleryImages || currentGalleryImages.length === 0) {
                alert("لا توجد صور في هذا العنصر");
                return;
            }
            // For the table, we show all element images since the images are structured by Element, not Director.
            activeLightboxImages = currentGalleryImages;
            currentGalleryIndex = 0;
            updateLightbox();
            document.getElementById('lightbox-overlay').style.display = 'flex';
        }

        function closeElementModal() { document.getElementById('element-details-modal').style.display = 'none'; }
        
        function openLightbox(idx) {
            activeLightboxImages = currentGalleryImages;
            currentGalleryIndex = idx;
            updateLightbox();
            document.getElementById('lightbox-overlay').style.display = 'flex';
        }
        function closeLightbox() { document.getElementById('lightbox-overlay').style.display = 'none'; }
        function changeLightboxImage(step, event) {
            event.stopPropagation();
            if (activeLightboxImages.length === 0) return;
            currentGalleryIndex += step;
            if (currentGalleryIndex >= activeLightboxImages.length) currentGalleryIndex = 0;
            if (currentGalleryIndex < 0) currentGalleryIndex = activeLightboxImages.length - 1;
            updateLightbox();
        }
        function updateLightbox() {
            if (activeLightboxImages.length === 0) return;
            const imgObj = activeLightboxImages[currentGalleryIndex];
            document.getElementById('lightbox-img').src = imgObj.url;
            document.getElementById('lightbox-caption').textContent = imgObj.caption + ` (${currentGalleryIndex + 1} / ${activeLightboxImages.length})`;
        }
        
        
                function handlePathHeaderClick(pathName) {
            if(currentStage === 'comparison') return; 
            const boxId = "path-box-" + btoa(unescape(encodeURIComponent(pathName))).replace(/=/g, '');
            selectedElementFilter = null; document.querySelectorAll('.element-row').forEach(r => r.classList.remove('active-element-filter'));
            if (selectedPathFilter === pathName) { 
                selectedPathFilter = null; document.getElementById(boxId).classList.remove('active-path-filter', 'open'); 
                renderGallery([]);
            } else { 
                document.querySelectorAll('.path-container').forEach(p => p.classList.remove('active-path-filter', 'open')); 
                selectedPathFilter = pathName; document.getElementById(boxId).classList.add('active-path-filter', 'open'); 
                if (currentStage === 'execution') {
                    let allImgs = [];
                    const pData = STAGES_DATA[currentStage].hierarchy[pathName];
                    if (pData) {
                        Object.keys(pData.elements).forEach(e => {
                            if(ELEMENT_IMAGES_EXEC[e]) allImgs = allImgs.concat(ELEMENT_IMAGES_EXEC[e]);
                        });
                    }
                    renderGallery(allImgs);
                } else {
                    renderGallery([]);
                }
            }
            refreshMapColors();
        }
                function handleElementRowClick(elementName, event) {
            event.stopPropagation();
            if(currentStage === 'comparison') return; 
            selectedPathFilter = null; document.querySelectorAll('.path-container').forEach(p => p.classList.remove('active-path-filter'));
            const rowId = "el-row-" + btoa(unescape(encodeURIComponent(elementName))).replace(/=/g, '');
            if (selectedElementFilter === elementName) { 
                selectedElementFilter = null; document.getElementById(rowId).classList.remove('active-element-filter'); 
                renderGallery([]);
            } else { 
                document.querySelectorAll('.element-row').forEach(r => r.classList.remove('active-element-filter')); selectedElementFilter = elementName; document.getElementById(rowId).classList.add('active-element-filter'); 
                if (currentStage === 'execution') {
                    const elImgs = ELEMENT_IMAGES_EXEC[elementName] || [];
                    renderGallery(elImgs);
                } else {
                    renderGallery([]);
                }
            }
            refreshMapColors();
        }
        function refreshMapColors() {
            let maxVal = 1; let targetCounts = {};
            if (currentStage === 'comparison') return;
            const stageInfo = STAGES_DATA[currentStage];
            if (!selectedElementFilter && !selectedPathFilter) {
                targetCounts = { ...stageInfo.counts }; maxVal = Math.max(...Object.values(targetCounts), 1);
                updateLegendUI("📊 التوزيع الجغرافي للمبادرات:", DEFAULT_SCALES, [Math.ceil(maxVal*0.25), Math.ceil(maxVal*0.5), Math.ceil(maxVal*0.75)]);
                Object.keys(REGION_AR).forEach(r => {
                    const count = targetCounts[r] || 0;
                    const fill = getHeatColorValue(count, maxVal, DEFAULT_SCALES);
                    document.querySelectorAll(`.gov-path[data-region="${r}"]`).forEach(p => p.setAttribute('fill', fill));
                    document.querySelectorAll(`text[data-region-text="${r}"]`).forEach(t => { t.setAttribute('fill', (count/maxVal) > 0.50 ? "#ffffff" : "#1e3d59"); });
                });
            } else if (selectedPathFilter) {
                const pElements = Object.keys(stageInfo.hierarchy[selectedPathFilter]["elements"]);
                const customScales = generateChoroplethScales(PATH_COLORS[selectedPathFilter] || "#b85c38");
                Object.keys(REGION_AR).forEach(r => { let total = 0; pElements.forEach(el => { stageInfo.events[r]?.forEach(ev => { if(ev.element === el) total++; }); }); targetCounts[r] = total; });
                maxVal = Math.max(...Object.values(targetCounts), 1);
                updateLegendUI(`📊 زخم مسار: ${selectedPathFilter}`, customScales, [Math.ceil(maxVal*0.25), Math.ceil(maxVal*0.5), Math.ceil(maxVal*0.75)]);
                Object.keys(REGION_AR).forEach(r => {
                    const count = targetCounts[r] || 0;
                    document.querySelectorAll(`.gov-path[data-region="${r}"]`).forEach(p => p.setAttribute('fill', getHeatColorValue(count, maxVal, customScales)));
                    document.querySelectorAll(`text[data-region-text="${r}"]`).forEach(t => t.setAttribute('fill', (count/maxVal) > 0.45 ? "#ffffff" : "#1e3d59"));
                });
            } else if (selectedElementFilter) {
                const customScales = generateChoroplethScales(ELEMENT_COLORS[selectedElementFilter] || "#b85c38");
                Object.keys(REGION_AR).forEach(r => { let c=0; stageInfo.events[r]?.forEach(ev => { if(ev.element === selectedElementFilter) c++; }); targetCounts[r] = c; });
                maxVal = Math.max(...Object.values(targetCounts), 1);
                updateLegendUI(`📊 زخم عنصر: ${selectedElementFilter}`, customScales, [Math.ceil(maxVal*0.25), Math.ceil(maxVal*0.5), Math.ceil(maxVal*0.75)]);
                Object.keys(REGION_AR).forEach(r => {
                    const count = targetCounts[r] || 0;
                    document.querySelectorAll(`.gov-path[data-region="${r}"]`).forEach(p => p.setAttribute('fill', getHeatColorValue(count, maxVal, customScales)));
                    document.querySelectorAll(`text[data-region-text="${r}"]`).forEach(t => t.setAttribute('fill', (count/maxVal) > 0.45 ? "#ffffff" : "#1e3d59"));
                });
            }
        }
        function getHeatColorValue(count, maxCount, scales) { if (count === 0) return scales[0]; const t = count / maxCount; if (t <= 0.25) return scales[1]; if (t <= 0.50) return scales[2]; if (t <= 0.75) return scales[3]; return scales[4]; }
        try {
            __GEOJSON_JS__
            const svg = document.getElementById('ksa-svg'); const NS = "http://www.w3.org/2000/svg"; const govGroups = {};
            Object.keys(REGION_AR).forEach(r => { const g = document.createElementNS(NS, 'g'); g.setAttribute('class', 'gov-group'); g.dataset.region = r; govGroups[r] = g; svg.appendChild(g); });
            if (typeof KSA_GEOJSON !== 'undefined' && KSA_GEOJSON.features) { KSA_GEOJSON.features.forEach(f => { const r = f.properties.name || f.properties.Name || f.properties.NAME || f.properties.name_1 || f.properties.NAME_1 || ""; if (!govGroups[r]) return; const geom = f.geometry; const polys = geom.type === 'Polygon' ? [geom.coordinates] : geom.coordinates; polys.forEach(poly => { const path = document.createElementNS(NS, 'path'); let d = ""; poly.forEach(ring => { ring.forEach((pt, i) => { const y = 33.0 - (pt[1] - 16.0); d += (i === 0 ? 'M' : 'L') + pt[0] + ',' + y; }); d += 'Z'; }); path.setAttribute('d', d); path.setAttribute('class', 'gov-path'); path.dataset.region = r; govGroups[r].appendChild(path); }); }); }
            Object.keys(govGroups).forEach(r => { 
                const g = govGroups[r]; const nameAr = REGION_AR[r] || r; 
                if (g.children.length > 0) { 
                    g.addEventListener('mouseenter', e => { 
                        if(!tip) return; 
                        if (currentStage === 'comparison') {
                            let execC = STAGES_DATA.execution.counts[r] || 0;
                            tip.innerHTML = `<strong>منطقة ${nameAr}</strong><br>تم تنفيذ <strong>${execC}</strong> مخرج إجمالاً في المنطقة`;
                        } else {
                            const stageInfo = STAGES_DATA[currentStage]; let count = 0; let totalFilterCount = 1;
                            if (selectedElementFilter) {
                                stageInfo.events[r]?.forEach(ev => { if(ev.element === selectedElementFilter) count++; });
                                totalFilterCount = 0; Object.values(stageInfo.events).forEach(reg => reg.forEach(ev => { if(ev.element===selectedElementFilter) totalFilterCount++; }));
                            } else if (selectedPathFilter) {
                                const pElements = Object.keys(stageInfo.hierarchy[selectedPathFilter]["elements"]);
                                stageInfo.events[r]?.forEach(ev => { if(pElements.includes(ev.element)) count++; });
                                totalFilterCount = 0; Object.values(stageInfo.events).forEach(reg => reg.forEach(ev => { if(pElements.includes(ev.element)) totalFilterCount++; }));
                            } else {
                                count = stageInfo.counts[r] || 0; totalFilterCount = stageInfo.total || 1;
                            }
                            if(totalFilterCount === 0) totalFilterCount = 1;
                            const pct = ((count / totalFilterCount) * 100).toFixed(1);
                            tip.innerHTML = `<strong>منطقة ${nameAr}</strong><br>📊 نسبة التغطية: <span style='color:#17b978; font-weight:800;'>${pct}%</span><br>🎯 المخرجات: ${count}`; 
                        }
                        tip.style.display = 'block'; moveTooltip(e); 
                    }); 
                    g.addEventListener('mousemove', moveTooltip); g.addEventListener('mouseleave', () => { if(tip) tip.style.display = 'none'; }); 
                    g.addEventListener('click', () => { if(currentStage !== 'comparison') handleRegionClick(r, nameAr); }); 
                    const pathBounds = g.getBBox(); const cx = pathBounds.x + pathBounds.width/2; const cy = pathBounds.y + pathBounds.height/2;
                    const t = document.createElementNS(NS, 'text'); t.setAttribute('x', cx); t.setAttribute('y', cy); t.setAttribute('text-anchor', 'middle'); t.setAttribute('font-size', '0.36'); t.setAttribute('font-weight', '800'); t.setAttribute('font-family', 'Tajawal'); t.setAttribute('pointer-events', 'none'); t.setAttribute('data-region-text', r); t.textContent = nameAr; svg.appendChild(t);
                } 
            });
        } catch (err) { console.error(err); }
        function moveTooltip(e) { if(!tip) return; const bounds = document.getElementById('map-panel').getBoundingClientRect(); tip.style.left = (e.clientX - bounds.left + 15) + 'px'; tip.style.top = (e.clientY - bounds.top - 45) + 'px'; }
        
        function handleRegionClick(regionKey, nameAr) { 
            currentOpenRegionKey = regionKey; 
            document.querySelectorAll('.gov-group').forEach(g => g.classList.remove('active-group')); 
            const selectedGroup = document.querySelector(`.gov-group[data-region="${regionKey}"]`); 
            if (selectedGroup) selectedGroup.classList.add('active-group'); 
            const stageInfo = STAGES_DATA[currentStage];
            document.getElementById('big-dashboard-modal').style.display = 'flex';
            let titleAddon = "";
            if (selectedElementFilter) titleAddon = ` (مفلتر لـ: ${selectedElementFilter})`;
            else if (selectedPathFilter) titleAddon = ` (مفلتر لـ: ${selectedPathFilter})`;
            document.getElementById('big-modal-title').textContent = "📍 التحليل لمنطقة " + nameAr + " - " + (currentStage==='preparation'?'الاعتماد':'التنفيذ') + titleAddon; 
            renderStageTimeline(regionKey, null); 
            const citiesContainer = document.getElementById('modal-cities-progress-list'); citiesContainer.innerHTML = ""; 
            let cityCounts = {};
            if (stageInfo.sub_charts[regionKey]) {
                Object.keys(stageInfo.sub_charts[regionKey]).forEach(c => { cityCounts[c] = 0; });
            }
            if (selectedElementFilter || selectedPathFilter) {
                let events = stageInfo.events[regionKey] || [];
                if (selectedElementFilter) { events = events.filter(e => e.element === selectedElementFilter); } 
                else if (selectedPathFilter) {
                    const pElements = Object.keys(stageInfo.hierarchy[selectedPathFilter]["elements"]);
                    events = events.filter(e => pElements.includes(e.element));
                }
                let cityIds = {}; Object.keys(cityCounts).forEach(c => cityIds[c] = new Set());
                events.forEach(e => {
                    let c = e.city || "-";
                    if (cityIds[c] !== undefined) { cityIds[c].add(e.id + "_" + e.element + "_" + e.director); }
                });
                Object.keys(cityIds).forEach(c => { cityCounts[c] = cityIds[c].size; });
            } else {
                cityCounts = stageInfo.sub_charts[regionKey] || {};
            }
            const subEntries = Object.entries(cityCounts).sort((a, b) => b[1] - a[1]); 
            const maxCityCount = subEntries.length > 0 ? Math.max(...subEntries.map(e => e[1])) : 1; 
            if (subEntries.length === 0) {
                 citiesContainer.innerHTML = `<div style="text-align:center; padding:20px; font-weight:800; color:#94a3b8;">لا توجد مخرجات للمدن بهذا الفلتر</div>`;
            } else {
                subEntries.forEach(([city, count]) => { 
                    const percentage = maxCityCount > 0 ? (count / maxCityCount) * 100 : 0; 
                    citiesContainer.innerHTML += `
                    <div class="city-progress-card" data-city="${city}" onclick="filterTimelineByCity('${city}')">
                        <div class="city-meta-line"><span>📍 ${city}</span><span>${count} مخرج</span></div>
                        <div class="progress-bar-bg">
                            <div class="progress-bar-fill" style="width: ${percentage}%; background: ${currentStage==='preparation'?'#1e3d59':'#b85c38'}"></div>
                        </div>
                    </div>`; 
                }); 
            }
        }
        function filterTimelineByCity(cityName) {
            document.getElementById('modal-timeline-title').textContent = "📅 سجل مخرجات مدينة: " + cityName;
            renderStageTimeline(currentOpenRegionKey, cityName);
            document.getElementById('clear-city-filter-btn').style.display = 'inline-block';
        }
        function clearCityFilter() {
            document.getElementById('modal-timeline-title').textContent = "📅 السجل الزمني التنفيذي للمخرجات";
            renderStageTimeline(currentOpenRegionKey, null);
            document.getElementById('clear-city-filter-btn').style.display = 'none';
        }
        function renderStageTimeline(regionKey, filterCity) { 
            const timelineDiv = document.getElementById('modal-events-timeline'); timelineDiv.innerHTML = ""; 
            const stageInfo = STAGES_DATA[currentStage];
            let events = stageInfo.events[regionKey] || [];
            if (filterCity) { events = events.filter(e => e.city === filterCity); }
            if (selectedElementFilter) { events = events.filter(e => e.element === selectedElementFilter); } 
            else if (selectedPathFilter) {
                const pElements = Object.keys(stageInfo.hierarchy[selectedPathFilter]["elements"]);
                events = events.filter(e => pElements.includes(e.element));
            }
            if(events.length === 0) {
                timelineDiv.innerHTML = `<div style="text-align:center; padding:30px; font-weight:800; color:#94a3b8;">لا توجد مخرجات مطابقة للفلتر الحالي في هذه المنطقة</div>`;
                return;
            }
            events.forEach(e => { let tagColor = ELEMENT_COLORS[e.element] || "#b85c38"; timelineDiv.innerHTML += `<div class="ev-card" style="border-right: 4px solid ${tagColor}"><div class="ev-header"><span class="ev-tag" style="background: ${tagColor}">🏷️ ${e.element}</span><span class="ev-date">📅 ${e.date}</span></div><div class="ev-title">🎬 مخرج: ${e.director}</div><div class="ev-desc">📝 الوصف: ${e.desc}</div><div class="ev-footer"><span>🏢 الجهة: ${e.org}</span><span>📍 المدينة: ${e.city}</span></div></div>`; }); 
        }
        function closeBigModal() { document.getElementById('big-dashboard-modal').style.display = 'none'; document.querySelectorAll('.gov-group').forEach(g => g.classList.remove('active-group')); }

        
        renderHierarchySidebar();
        refreshMapColors();
        triggerStageChange('preparation');
    </script>

"""
final_html = html_content + html_script
final_html = final_html.replace("__PREP_DATA_JSON__", json.dumps(data_preparation, ensure_ascii=False))
final_html = final_html.replace("__KPI_TARGETS_JSON__", json.dumps(kpi_targets, ensure_ascii=False))
final_html = final_html.replace("__EXEC_DATA_JSON__", json.dumps(data_execution, ensure_ascii=False))
final_html = final_html.replace("__ELEMENT_IMAGES_PREP_JSON__", json.dumps(element_images_prep, ensure_ascii=False))
final_html = final_html.replace("__ELEMENT_IMAGES_EXEC_JSON__", json.dumps(element_images_exec, ensure_ascii=False))

In [ ]:
geojson_js_val = ""
if SHP_JS_PATH.exists():
    with open(SHP_JS_PATH, "r", encoding="utf-8") as f:
        geojson_js_val = f.read()

In [ ]:
final_html = final_html.replace("__GEOJSON_JS__", geojson_js_val)
final_html = final_html.replace("__REGIONS_MAP_AR__", json.dumps(REGION_TO_ARABIC, ensure_ascii=False))
final_html = final_html.replace("__PATH_COLORS_JSON__", json.dumps(path_colors_map, ensure_ascii=False))
final_html = final_html.replace("__ELEMENT_COLORS_JSON__", json.dumps(element_colors_map, ensure_ascii=False))

In [ ]:
with open(OUT_HTML, "w", encoding="utf-8") as f: 
    f.write(final_html)

In [ ]:
try:
    webbrowser.open(Path(OUT_HTML).resolve().as_uri())
except Exception as e:
    print(f"Could not open browser: {e}")
print("✅ تم كتابة واستخراج ملف HTML بنجاح!")